# Classification over 255 classes

Other decision APIs take at most 255 options in one choice. With TypeLLM's latest hosted model, one choice takes up to 512, each with its probability and a confidence.

Time zones need more than 255. An email says "3pm my time, I'm in Saskatoon." To book the meeting you need the sender's IANA time zone, and there are 418 of them. Saskatoon is `America/Regina`: Saskatchewan skips daylight saving, so a coarse "Mountain" or "Central" puts the meeting an hour off for half the year.

All 418 time zones go in one choice, answered in one API call.

1. Load the time zones.
2. Find each sender's time zone in one API call.

Paste your TypeLLM API key in the cell below, then run all cells.

In [1]:
%pip install -q "typellm>=0.6.9" tzdata

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Paste your API key here.
TYPELLM_API_KEY = ""  # https://typellm.ai/dashboard/keys

In [3]:
from typellm import TypeLLMClient

client = TypeLLMClient(api_key=TYPELLM_API_KEY)

EMAILS = [
    "Thursday 9am works for me. I'm in Porto Alegre this month.",
    "Can we do 3pm my time? I'm in Saskatoon.",
    "Morning works best. Our office is in Bangalore.",
    "Can we do 3pm my time? I'm based in Scottsdale.",
    "Let's talk at 10. I'm in Indiana.",
]

## Load the time zones

`zone.tab` from the `tzdata` package lists the 418 time zones in use, one per region.

In [4]:
from importlib.resources import files

lines = files("tzdata.zoneinfo").joinpath("zone.tab").read_text().splitlines()
ZONES = sorted(line.split("\t")[2] for line in lines if line and not line.startswith("#"))
len(ZONES)

418

## Find each sender's time zone in one call

All 418 time zones are the options of one choice. `return_probabilities` returns a probability for each, and a confidence.

In [5]:
QUESTIONS = {"time_zone": {
    "type": "string", "enum": ZONES, "return_probabilities": True,
    "instructions": "Which IANA time zone is the sender in?",
}}

for email in EMAILS:
    answer = client.generate(context=f"Email: {email}", questions=QUESTIONS).result["time_zone"]
    print(f"{answer['value']:30s} {answer['confidence']:.2f}  <- {email}")

America/Sao_Paulo              1.00  <- Thursday 9am works for me. I'm in Porto Alegre this month.


America/Regina                 1.00  <- Can we do 3pm my time? I'm in Saskatoon.


Asia/Kolkata                   1.00  <- Morning works best. Our office is in Bangalore.


America/Phoenix                1.00  <- Can we do 3pm my time? I'm based in Scottsdale.


America/Indiana/Indianapolis   1.00  <- Let's talk at 10. I'm in Indiana.
